# Tema V - Ingeniería de Características (FE)

## Objetivos de Aprendizaje

Al finalizar este cuaderno serás capaz de:

- Imputar Valores Ausentes de forma Avanzada.
- Codificar Variables Categóricas
- Transformar y Escalar variables.
- Discretizar y mezclar variables (binning).
- Extraer y Crear nuevas características.
- Reorganizar y Balancear Datos.
___


## Imputación Avanzada de Valores Ausentes

### Objetivos

Al finalizar este apartado serás capaz de:

* Superar las limitaciones de la imputación global simple (mean(), median()).
* Implementar imputaciones condicionales por grupos utilizando el patrón groupby() + transform().
* Capturar el patrón de falta de información mediante variables indicadoras (Missingness Indicators).

### Imputación Condicional por Grupos

Imputar el valor medio o la mediana de todo el dataset suele distorsionar la distribución de los datos. La imputación por grupos utiliza el contexto de otras variables categóricas para estimar un valor sustituto mucho más preciso.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.datasets import make_classification
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek

In [2]:
# Dataset con salarios ausentes por departamento
df = pd.DataFrame({
    "Departamento": ["IT", "IT", "IT", "Ventas", "Ventas", "Ventas"],
    "Experiencia": ["Senior", "Junior", "Senior", "Senior", "Junior", "Junior"],
    "Salario": [60000, 35000, np.nan, 45000, 25000, np.nan]
})

# Imputación condicional: Mediana del Salario según el Departamento
df["Salario_Imp_Depto"] = df.groupby("Departamento")["Salario"].transform(
    lambda x: x.fillna(x.median())
)

# Imputación condicional fina: Mediana según Departamento Y Experiencia
df["Salario_Imp_Fina"] = df.groupby(["Departamento", "Experiencia"])["Salario"].transform(
    lambda x: x.fillna(x.median())
)

print(df[["Departamento", "Experiencia", "Salario", "Salario_Imp_Fina"]])

  Departamento Experiencia  Salario  Salario_Imp_Fina
0           IT      Senior  60000.0           60000.0
1           IT      Junior  35000.0           35000.0
2           IT      Senior      NaN           60000.0
3       Ventas      Senior  45000.0           45000.0
4       Ventas      Junior  25000.0           25000.0
5       Ventas      Junior      NaN           25000.0


### Indicadores de Ausencia (Missingness Indicators)

En muchos escenarios de negocio, el hecho de que un dato no exista no es aleatorio (Not Missing at Random - NMAR); contiene un significado operativo (ej. un cliente que no rellena su nivel de ingresos o no responde a una encuesta).

Crear una columna booleana que señale si el dato original estaba ausente preserva esta señal predictiva para los modelos del Tema VI.

In [3]:
# Crear variable indicadora booleana (1 si era nulo, 0 si no)
df["Salario_EsNulo"] = df["Salario"].isna().astype(int)

# Tras guardar la señal, se procede a la imputación de la columna original
df["Salario"] = df["Salario"].fillna(df["Salario"].median())

print(df[["Salario", "Salario_EsNulo"]])

   Salario  Salario_EsNulo
0  60000.0               0
1  35000.0               0
2  40000.0               1
3  45000.0               0
4  25000.0               0
5  40000.0               1


### Ideas Clave

- Preservación de la varianza: La imputación condicional (groupby + transform) respeta la estructura subyacente de los subgrupos, evitando la reducción artificial de varianza que produce la imputación global.
- Información implícita: La falta de un dato puede ser una característica altamente predictiva por sí misma; la columna indicadora garantiza que no se pierda esa información al imputar.
- Riesgo de grupos pequeños: Al imputar por múltiples subgrupos, se debe verificar que todos los grupos contengan al menos un valor no nulo para evitar propagar los NaN.
___

## Codificación de Variables Categóricas (Encoding)

### Objetivos

Al finalizar este apartado serás capaz de:

* Comprender por qué los algoritmos numéricos requieren la transformación de variables categóricas.
* Aplicar Mapeo Ordinal respetando la jerarquía intrínseca de los datos.
* Implementar One-Hot Encoding en variables nominales y mitigar la "trampa de las variables ficticias" (Dummy Variable Trap).
* Introducir el Target Encoding para categóricas con alta cardinalidad.

### Codificación Ordinal (Ordinal Encoding)

Se utiliza cuando las categorías poseen un orden o jerarquía natural (ej. Nivel Educativo, Calificación de Riesgo). Asigna un entero secuencial preservando la relación de orden ($0 < 1 < 2 < \dots$).

In [4]:
df = pd.DataFrame({
    "Nivel_Estudios": ["Secundaria", "Grado", "Master", "Doctorado", "Grado"],
    "Satisfaccion": ["Baja", "Media", "Alta", "Alta", "Baja"]
})

# Mapeo explícito mediante diccionario
mapa_estudios = {
    "Secundaria": 0,
    "Grado": 1,
    "Master": 2,
    "Doctorado": 3
}

df["Nivel_Estudios_Encoded"] = df["Nivel_Estudios"].map(mapa_estudios)
print(df[["Nivel_Estudios", "Nivel_Estudios_Encoded"]])

  Nivel_Estudios  Nivel_Estudios_Encoded
0     Secundaria                       0
1          Grado                       1
2         Master                       2
3      Doctorado                       3
4          Grado                       1


### Codificación One-Hot (One-Hot Encoding)

Se aplica a variables nominales sin orden jerárquico (ej. Ciudad, Método de Pago). Crea una nueva columna binaria ($0$ o $1$) por cada categoría presente.

Si una variable tiene $K$ categorías, la $K$-ésima columna es linealmente dependiente de las primeras $K-1$. Para evitar la colinealidad perfecta en modelos lineales, se suele eliminar la primera columna (drop_first=True).

In [5]:
df_pagos = pd.DataFrame({
    "Metodo_Pago": ["Tarjeta", "Transferencia", "Efectivo", "Tarjeta"]
})

# Aplicación con Pandas pd.get_dummies()
df_dummies = pd.get_dummies(
    df_pagos, 
    columns=["Metodo_Pago"], 
    drop_first=True,  # Evita multicolinealidad perfecta
    dtype=int
)
print(df_dummies)

   Metodo_Pago_Tarjeta  Metodo_Pago_Transferencia
0                    1                          0
1                    0                          1
2                    0                          0
3                    1                          0


### Codificación orientada a la Variable Objetivo (Target Encoding)

Cuando una variable nominal tiene alta cardinalidad (ej. Código Postal con 500 categorías), el One-Hot Encoding genera demasiadas columnas sparse (matriz rala).

El Target Encoding reemplaza cada categoría por la media de la variable objetivo (y) dentro de ese grupo.

In [6]:
df_postal = pd.DataFrame({
    "Codigo_Postal": ["28001", "28001", "08001", "08001", "41001"],
    "Cancelado": [1, 0, 1, 1, 0]  # Variable objetivo binary
})

# Cálculo del promedio del Target por grupo
target_map = df_postal.groupby("Codigo_Postal")["Cancelado"].mean().to_dict()

# Mapeo directo sobre el DataFrame
df_postal["CP_Target_Encoded"] = df_postal["Codigo_Postal"].map(target_map)
print(df_postal)

  Codigo_Postal  Cancelado  CP_Target_Encoded
0         28001          1                0.5
1         28001          0                0.5
2         08001          1                1.0
3         08001          1                1.0
4         41001          0                0.0


**Advertencia de Seguridad:** En entornos de producción o modelado, el Target Encoding debe calcularse únicamente sobre el conjunto de entrenamiento (Train set) y mapearse al de prueba (Test set) para prevenir la fuga de datos (Data Leakage).

### Ideas Clave

* Elegir la técnica según la naturaleza: Usar Ordinal Encoding solo si hay jerarquía real. Aplicar enteros arbitrarios a variables nominales engaña al modelo (asume que "Madrid" (2) es el doble que "Barcelona" (1)).
* Uso de drop_first=True: Es obligatorio en regresiones y modelos basados en matrices para evitar redundancia lineal matemática.
* Control de dimensionalidad: Si One-Hot Encoding crea más de 20-30 columnas nuevas, es momento de considerar Target Encoding o agrupamiento previo de categorías minoritarias en un grupo "Otros".
___

## Transformación y Escalado de Variables Numéricas

### Objetivos

Al finalizar este apartado serás capaz de:

* Comprender la sensibilidad de los algoritmos basados en distancia (KNN, SVM, K-Means) o degradado a las distintas escalas numéricas.
* Aplicar Estandarización (StandardScaler) y Escalado Min-Max (MinMaxScaler).
* Corregir el sesgo y las colas pesadas mediante transformaciones logarítmicas.

### Introducción

Cuando las variables numéricas conviven en rangos significativamente dispares (ej. Edad entre 18 y 85 años frente a Ingresos_Anuales entre 15.000 y 120.000 €), la variable con magnitudes mayores dominará los cálculos de distancia e influencia en los gradientes, distorsionando el aprendizaje del algoritmo.

### Métodos Principales de Escalado

|Método|  Fórmula  |Rango resultante|Sensibilidad a Outliers|Uso principal|
|------|----------------------------|----------------|-----------------------|-------------|
|StandardScaler (Z-Score)|$x'=(x−μ)/σ​$|Centrado en μ=0, σ=1|Moderada|Algoritmos lineales, SVM, PCA, Regresión Logística.|
|MinMaxScaler|$x'=(xmax​−xmin​)/(x−xmin)​$|​Fijo entre [0,1]|Muy Alta|Redes Neuronales, KNN, algoritmos con entradas acotadas.|
|RobustScaler|$x'=(x−Mediana)/IQR$|​Centrado en la mediana|Muy Baja (Robusto)|Datasets con presencia fuerte de outliers.|

In [7]:
df = pd.DataFrame({
    "Edad": [20, 40, 50, 25, 60],
    "Ingresos": [18000, 45000, 80000, 22000, 120000]
})

# 1. Estandarización (StandardScaler)
scaler_std = StandardScaler()
df_std = pd.DataFrame(scaler_std.fit_transform(df), columns=["Edad_Std", "Ingresos_Std"])

# 2. Escalado Normalizado (MinMaxScaler)
scaler_minmax = MinMaxScaler()
df_minmax = pd.DataFrame(scaler_minmax.fit_transform(df), columns=["Edad_MinMax", "Ingresos_MinMax"])

df_escalado = pd.concat([df, df_std, df_minmax], axis=1)
print(df_escalado)

   Edad  Ingresos  Edad_Std  Ingresos_Std  Edad_MinMax  Ingresos_MinMax
0    20     18000 -1.269491     -1.014580        0.000         0.000000
1    40     45000  0.066815     -0.312178        0.500         0.264706
2    50     80000  0.734968      0.598342        0.750         0.607843
3    25     22000 -0.935414     -0.910520        0.125         0.039216
4    60    120000  1.403122      1.638936        1.000         1.000000


### Transformaciones para Asimetría (Log Transform)

Las variables financieras, de consumo o de recuento suelen presentar un marcado sesgo a la derecha (right-skewed). La transformación logarítmica comprime las colas largas y aproxima la distribución a una forma Normal.

$$\tilde{x} = \log(1 + x)$$

In [8]:
# Se utiliza np.log1p(x) para evitar errores matemáticos si x = 0 (calcula log(1 + x))
df["Ingresos_Log"] = np.log1p(df["Ingresos"])

print("Asimetría original:", df["Ingresos"].skew())
print("Asimetría tras Log:", df["Ingresos_Log"].skew())

Asimetría original: 0.8308973646347954
Asimetría tras Log: 0.09182475799532626


### Ideas Clave

- Fit en Train, Transform en Test: Los transformadores (fit_transform) deben ajustar sus parámetros ($\mu, \sigma, min, max$) únicamente con los datos de entrenamiento para prevenir el Data Leakage.
- Efecto nulo en árboles: Los algoritmos basados en árboles de decisión (Random Forest, XGBoost) son inmunes a las transformaciones monótonas de escala; no requieren escalado previo.
- log1p vs log: Utilizar siempre np.log1p() en lugar de np.log() para prevenir indeterminaciones matemáticas cuando la variable contiene valores cero.
___

## Discretización y Agrupamiento en Intervalos (Binning)

### Objetivos

Al finalizar este apartado serás capaz de:

* Convertir variables numéricas continuas en categorías ordenadas para simplificar patrones complejos.
* Implementar discretización por ancho de intervalo fijo (pd.cut()).
* Implementar discretización por cuantiles/frecuencia fija (pd.qcut()).
* Analizar cuándo el binning mejora la interpretabilidad de negocio sin perder información crítica.

### Introducción

La discretización o binning transforma una variable continua en intervalos discretos (cubos o bins). Esta técnica es útil en los siguientes escenarios:

* Efectos no lineales por tramos: El impacto de una variable no cambia de forma lineal, sino por umbrales (ej. el riesgo de un préstamo no escala punto a punto con la edad, sino por franjas de edad).
* Manejo de ruido y valores atípicos: Agrupar en tramos suaviza pequeñas fluctuaciones o outliers en los extremos.
* Interpretabilidad de negocio: Facilita la creación de segmentos operativos (ej. "Consumo Bajo", "Consumo Medio", "Consumo Alto").

### Discretización de Ancho Fijo (pd.cut()) vs. Cuantiles (pd.qcut())

Existen dos estrategias principales para definir los límites de los intervalos:

|Método|Función en Pandas|Lógica de división|Resultado en los intervalos|
|------|-----------------|------------------|---------------------------|
|Ancho Fijo|pd.cut()|Divide el rango total [xmin​,xmax​] en N intervalos de igual amplitud.|Los tramos tienen el mismo tamaño, pero el número de registros por tramo puede ser muy desequilibrado.|
|Frecuencia Fija (Cuantiles)|pd.qcut()|Divide los datos en N intervalos asegurando que cada uno contenga el mismo número de observaciones.|La amplitud de cada tramo varía, pero el número de registros por tramo es casi idéntico (percentiles).|

In [9]:
# Dataset de prueba: Edad de 10 clientes
df = pd.DataFrame({
    "Edad": [18, 20, 22, 25, 30, 35, 40, 50, 65, 80]
})

# 1. pd.cut(): Definición de bordes explícitos con etiquetas de negocio
etiquetas_edad = ["Joven", "Adulto", "Senior"]
bordes = [17, 30, 55, 100]  # (17, 30], (30, 55], (55, 100]

df["Tramo_Edad_Custom"] = pd.cut(
    df["Edad"], 
    bins=bordes, 
    labels=etiquetas_edad, 
    include_lowest=True
)

# 2. pd.qcut(): División por tertiles (3 grupos con el mismo número de elementos)
df["Tramo_Edad_Cuantil"] = pd.qcut(
    df["Edad"], 
    q=3, 
    labels=["Bajo", "Medio", "Alto"]
)

print(df)

   Edad Tramo_Edad_Custom Tramo_Edad_Cuantil
0    18             Joven               Bajo
1    20             Joven               Bajo
2    22             Joven               Bajo
3    25             Joven               Bajo
4    30             Joven              Medio
5    35            Adulto              Medio
6    40            Adulto              Medio
7    50            Adulto               Alto
8    65            Senior               Alto
9    80            Senior               Alto


### Inspección del Equilibrio entre Métodos

Es fundamental auditar el conteo de registros por cada tramo generado para evitar categorías vacías o hiper-representadas:

In [10]:
print("--- Conteo con pd.cut (Ancho/Límites definidos) ---")
print(df["Tramo_Edad_Custom"].value_counts())

print("\n--- Conteo con pd.qcut (Misma frecuencia por grupo) ---")
print(df["Tramo_Edad_Cuantil"].value_counts())

--- Conteo con pd.cut (Ancho/Límites definidos) ---
Tramo_Edad_Custom
Joven     5
Adulto    3
Senior    2
Name: count, dtype: int64

--- Conteo con pd.qcut (Misma frecuencia por grupo) ---
Tramo_Edad_Cuantil
Bajo     4
Medio    3
Alto     3
Name: count, dtype: int64


### Ideas Clave

- Elección basada en la distribución: Si los datos tienen un sesgo muy marcado (skewed), pd.cut() agrupará casi todos los registros en el primer tramo. En esos casos, pd.qcut() es más efectivo para equilibrar las muestras.
- Pérdida de resolución: La discretización simplifica los datos, pero elimina la variabilidad fina dentro de cada tramo. Se debe usar con moderación en algoritmos que aprovechan bien la continuidad matemática (como modelos lineales o SVM).
- Manejo de bordes en producción: Al aplicar pd.cut() en nuevos datos (test set o producción), valores fuera del rango $[x_{min}, x_{max}]$ definido en train se convertirán en NaN. Define siempre márgenes holgados en los límites exterior e inferior.
___

## Extracción y Creación de Nuevas Características (Feature Extraction / Domain-Specific Features)

### Objetivos

Al finalizar este apartado será capaz de:

* Generar nuevas variables sintéticas a partir de conocimiento del dominio del negocio (Domain Knowledge).
* Crear ratios e interacciones matemáticas que simplifiquen las relaciones no lineales para los modelos.
* Extraer componentes temporales (año, mes, día de la semana) y calcular métricas de antigüedad o frecuencia.
* Representar variables temporales cíclicas mediante transformaciones trigonométricas (seno/coseno).

### Introducción

A menudo, la combinación matemática de dos o más variables crudas aporta un valor predictivo mucho mayor que el de las variables por separado.

- Ratios de consumo/eficiencia: Expresan relaciones de intensidad (ej. gasto por mes, uso por dólar pagado).

- Interacciones multiplicativas: Representan efectos combinados entre dos magnitudes (ej. $\text{Antigüedad} \times \text{Cargos Mensuales}$ = Valor histórico estimado del cliente).

In [11]:
df = pd.DataFrame({
    "Antiguedad_Meses": [12, 24, 6, 48],
    "Cargos_Mensuales": [50.0, 80.0, 30.0, 100.0],
    "Soporte_Tickets": [6, 2, 8, 1]
})

# 1. Ratio: Promedio de tickets generados por mes
df["Tickets_Por_Mes"] = df["Soporte_Tickets"] / df["Antiguedad_Meses"]

# 2. Interacción: Estimación del valor de vida acumulado aproximado (CLV simplificado)
df["Valor_Total_Estimado"] = df["Antiguedad_Meses"] * df["Cargos_Mensuales"]

print(df[["Tickets_Por_Mes", "Valor_Total_Estimado"]])

   Tickets_Por_Mes  Valor_Total_Estimado
0         0.500000                 600.0
1         0.083333                1920.0
2         1.333333                 180.0
3         0.020833                4800.0


### Extracción de Características Temporales

Las marcas de tiempo (timestamps) no pueden introducirse directamente en un modelo. Es necesario descomponerlas en componentes numéricos explicativos:

In [12]:
df_ventas = pd.DataFrame({
    "Fecha_Registro": pd.to_datetime(["2025-01-15 08:30:00", "2025-06-20 18:45:00", "2025-12-01 12:00:00"])
})

# Extracción de componentes directos
df_ventas["Año"] = df_ventas["Fecha_Registro"].dt.year
df_ventas["Mes"] = df_ventas["Fecha_Registro"].dt.month
df_ventas["Dia_Semana"] = df_ventas["Fecha_Registro"].dt.dayofweek # 0 = Lunes, 6 = Domingo
df_ventas["Es_Fin_De_Semana"] = (df_ventas["Dia_Semana"] >= 5).astype(int)

# Cálculo de recesión o antigüedad respecto a un punto de corte
fecha_referencia = pd.to_datetime("2026-01-01")
df_ventas["Dias_Desde_Registro"] = (fecha_referencia - df_ventas["Fecha_Registro"]).dt.days

print(df_ventas[["Mes", "Dia_Semana", "Es_Fin_De_Semana", "Dias_Desde_Registro"]])

   Mes  Dia_Semana  Es_Fin_De_Semana  Dias_Desde_Registro
0    1           2                 0                  350
1    6           4                 0                  194
2   12           0                 0                   30


### Transformación Trigonométrica de Variables Cíclicas

Las variables como el mes del año (1-12) o la hora del día (0-23) son periódicas: la hora 23 y la hora 0 están separadas por un solo 60 minutos, pero numéricamente parecen estar en extremos opuestos.

Mapear estas variables sobre un círculo mediante Seno y Coseno permite que el modelo entienda la continuidad del ciclo.

$$\text{Mes\_Sin} = \sin\left(\frac{2 \pi \times \text{Mes}}{12}\right), \quad \text{Mes\_Cos} = \cos\left(\frac{2 \pi \times \text{Mes}}{12}\right)$$

In [13]:
# Transformación cíclica para el Mes (1-12)
df_ventas["Mes_Sin"] = np.sin(2 * np.pi * df_ventas["Mes"] / 12)
df_ventas["Mes_Cos"] = np.cos(2 * np.pi * df_ventas["Mes"] / 12)

print(df_ventas[["Mes", "Mes_Sin", "Mes_Cos"]])

   Mes       Mes_Sin   Mes_Cos
0    1  5.000000e-01  0.866025
1    6  1.224647e-16 -1.000000
2   12 -2.449294e-16  1.000000


### Ideas Clave

- Inspiración en el negocio: La creación de features efectivas proviene de entender el problema, no solo de aplicar operaciones matemáticas al azar.
- Evitar división por cero: Al construir ratios (ej. $\frac{A}{B}$), siempre hay que proteger el código contra denominadores nulos o ceros mediante .replace(0, np.nan) o sumando una pequeña constante $\epsilon$.
- Continuidad cíclica: La transformación Seno/Coseno es indispensable para algoritmos sensibles a distancias (KNN, Redes Neuronales, SVM) cuando operan con fechas, horas o ángulos.
___

## Reorganización y Balanceo de Datos (Imbalanced Data & Resampling)

### Objetivos

* Comprender las consecuencias del desbalanceo de clases en los algoritmos de clasificación.
* Aplicar técnicas de submuestreo (Undersampling) y sobremuestreo (Oversampling) sintético.
* Generar muestras sintéticas mediante el algoritmo SMOTE (Synthetic Minority Over-sampling Technique).
* Implementar la limpieza de fronteras mediante Tomek Links y preparar el dataset final para el Tema VI.

### El Problema de las Clases Desequilibradas

En problemas reales como la predicción de churn, la clase de interés (1 = Canceló) suele representar solo entre el $5\%$ y el $20\%$ del total. Si un modelo se entrena con este desbalanceo, alcanzará un $90\%$ de precisión (Accuracy) simplemente prediciendo siempre la clase mayoritaria (0 = Activo), sin aprender a detectar los clientes en riesgo.

### Estrategias de Reorganización de Muestras

                    [Dataset Desbalanceado]
                               │
         ┌─────────────────────┴─────────────────────┐
         ▼                                           ▼
    [Undersampling (Submuestreo)]             [Oversampling (Sobremuestreo)]
    Elimina datos de la clase mayoritaria.   Genera muestras de la clase minoritaria.
      • Ejemplo: Random UnderSampler           • Ejemplo: SMOTE (Sintético)
      • Riesgo: Pérdida de información.        • Riesgo: Overfitting si es aleatorio simple.

### Generación Sintética con SMOTE y Limpieza con Tomek Links

A diferencia de la duplicación simple de filas, SMOTE busca los $k$-vecinos más cercanos de la clase minoritaria e interpola matemáticamente nuevos puntos a lo largo de las líneas que los unen.

In [14]:
# 1. Crear dataset sintético desbalanceado (90% Clase 0, 10% Clase 1)
X, y = make_classification(
    n_samples=1000, n_features=4, n_clusters_per_class=1,
    weights=[0.9, 0.1], random_state=42
)

print(f"Distribución original: {pd.Series(y).value_counts().to_dict()}")

# 2. Aplicación de SMOTE (Sobremuestreo Sintético)
smote = SMOTE(sampling_strategy="auto", random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)
print(f"Tras SMOTE: {pd.Series(y_smote).value_counts().to_dict()}")

# 3. Estrategia Híbrida: SMOTE + Tomek Links (Genera sintéticos y limpia bordes ruidosos)
smt = SMOTETomek(random_state=42)
X_resampled, y_resampled = smt.fit_resample(X, y)
print(f"Tras SMOTE + Tomek Links: {pd.Series(y_resampled).value_counts().to_dict()}")

Distribución original: {0: 892, 1: 108}
Tras SMOTE: {0: 892, 1: 892}
Tras SMOTE + Tomek Links: {0: 858, 1: 858}


### Ideas Clave

- Regla de oro del Balanceo: El balanceo de datos (SMOTE o Undersampling) debe aplicarse exclusivamente sobre el conjunto de entrenamiento (Train set). Aplicarlo al conjunto de prueba (Test set) contamina las métricas de evaluación con datos sintéticos irrealistas.
- SMOTE requiere datos numéricos: SMOTE calcula distancias euclídeas entre puntos; por ello, debe aplicarse después de haber completado la codificación (Encoding) y el escalado de características.
- Tomek Links para limpiar fronteras: Elimina parejas de puntos opuestos muy cercanos en el espacio de características, suavizando la frontera de decisión para que los clasificadores del Tema VI no memoricen ruido.
- Tránsito al Tema VI: Con este bloque concluimos el Tema V. El alumno dispone ahora de un pipeline capaz de transformar un dataset sucio y crudo en una matriz puramente numérica, escalada y estadísticamente equilibrada.
___